# Clase 12 — Archivos y Excepciones

En esta clase aprenderemos a **leer y escribir archivos** en Python, y a **manejar errores** con el bloque try/except. Estas son habilidades fundamentales para cualquier programa real.

**Contenido de esta clase:**
1. Apertura de archivos
2. Lectura de archivos
3. Escritura de archivos
4. Administradores de contexto (`with`)
5. Excepciones: try/except/finally
6. Lanzar excepciones (`raise`)
7. Excepciones personalizadas
8. Ejercicio práctico


## 1. Apertura de archivos

Python usa la función `open()` para abrir archivos. Los modos más comunes son:

| Modo | Descripción |
|---|---|
| `'r'` | Lectura (default) |
| `'w'` | Escritura (sobrescribe) |
| `'a'` | Append (agrega al final) |
| `'x'` | Exclusivo (falla si existe) |
| `'b'` | Modo binario |
| `'t'` | Modo texto (default) |


In [ ]:
# Modos de apertura
# 'r' = lectura, 'w' = escritura, 'a' = append, 'r+' = lectura/escritura

# Ejemplo: crear un archivo temporal para pruebas
import os
import tempfile

# Crear un archivo temporal
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as f:
    f.write("Línea 1\nLínea 2\nLínea 3\n")
    ruta_temp = f.name

print(f"Archivo creado: {ruta_temp}")
print(f"¿Existe? {os.path.exists(ruta_temp)}")


## 2. Lectura de archivos


In [ ]:
# Leer todo el contenido de una vez
with open(ruta_temp, 'r') as archivo:
    contenido = archivo.read()
print("Contenido completo:")
print(contenido)


In [ ]:
# Leer línea por línea
print("Línea por línea:")
with open(ruta_temp, 'r') as archivo:
    for i, linea in enumerate(archivo, 1):
        print(f"  {i}: {linea.rstrip()}")


In [ ]:
# Leer todas las líneas en una lista
with open(ruta_temp, 'r') as archivo:
    lineas = archivo.readlines()
print(f"Tipo: {type(lineas).__name__}")
print(f"Número de líneas: {len(lineas)}")
print(f"Primera línea: {lineas[0].rstrip()}")


In [ ]:
# Leer solo las primeras N líneas
with open(ruta_temp, 'r') as archivo:
    primera = archivo.readline()
    segunda = archivo.readline()
print(f"L1: {primera.rstrip()}")
print(f"L2: {segunda.rstrip()}")


## 3. Escritura de archivos


In [ ]:
# Crear un nuevo archivo y escribir
ruta_nueva = os.path.join(tempfile.gettempdir(), "mi_archivo.txt")

with open(ruta_nueva, 'w', encoding='utf-8') as archivo:
    archivo.write("Primera línea\n")
    archivo.write("Segunda línea\n")
    archivo.write("Tercera línea\n")

print(f"Archivo escrito: {ruta_nueva}")

# Leer para verificar
with open(ruta_nueva, 'r', encoding='utf-8') as archivo:
    print(archivo.read())


In [ ]:
# Escribir múltiples líneas de una vez
lineas = ["Manzana\n", "Pera\n", "Naranja\n", "Uva\n"]

with open(ruta_nueva, 'w', encoding='utf-8') as archivo:
    archivo.writelines(lineas)

# Verificar
with open(ruta_nueva, 'r', encoding='utf-8') as archivo:
    print(archivo.read())


In [ ]:
# Append: agregar al final sin sobrescribir
with open(ruta_nueva, 'a', encoding='utf-8') as archivo:
    archivo.write("Kiwi\n")

with open(ruta_nueva, 'r', encoding='utf-8') as archivo:
    print(archivo.read())


## 4. Administradores de contexto (`with`)

El bloque `with` garantiza que el archivo se cierre automáticamente, incluso si ocurre un error.


In [ ]:
# Comparación: con y sin 'with'
# ❌ Sin 'with': riesgo de no cerrar el archivo
archivo = open(ruta_nueva, 'r')
contenido = archivo.read()
# Si olvidas cerrar, hay problemas de memoria y bloqueo de archivos
archivo.close()


In [ ]:
# ✓ Con 'with': cierre automático garantizado
with open(ruta_nueva, 'r', encoding='utf-8') as archivo:
    contenido = archivo.read()
    # No necesitas llamar a close()
# El archivo ya está cerrado fuera del bloque


In [ ]:
# Ejemplo: procesar un archivo CSV simple
import csv

# Escribir un CSV
ruta_csv = os.path.join(tempfile.gettempdir(), "datos.csv")
with open(ruta_csv, 'w', newline='', encoding='utf-8') as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(['Nombre', 'Edad', 'Ciudad'])
    escritor.writerow(['Ana', 25, 'CDMX'])
    escritor.writerow(['Luis', 30, 'Madrid'])

# Leer el CSV
with open(ruta_csv, 'r', encoding='utf-8') as archivo:
    lector = csv.reader(archivo)
    for fila in lector:
        print(fila)


## 5. Excepciones: try/except/finally


In [ ]:
# Estructura básica de try/except
try:
    numero = int(input("Ingresa un número: "))
    print(f"Ingresaste: {numero}")
except ValueError:
    print("¡Error! No ingresaste un número válido.")


In [ ]:
# Capturar múltiples excepciones
def dividir(a, b):
    try:
        resultado = a / b
    except ZeroDivisionError:
        return "Error: división por cero"
    except TypeError:
        return "Error: tipos incompatibles"
    return resultado

print(f"10 / 2 = {dividir(10, 2)}")
print(f"10 / 0 = {dividir(10, 0)}")
print(f"10 / 'a' = {dividir(10, 'a')}")


In [ ]:
# Capturar cualquier excepción
def operacion_segura(a, b):
    try:
        resultado = a / b
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"
    return resultado

print(f"10 / 0 = {operacion_segura(10, 0)}")
print(f"10 / 'a' = {operacion_segura(10, 'a')}")


In [ ]:
# try/except/else/finally completo
def leer_archivo(ruta):
    try:
        archivo = open(ruta, 'r', encoding='utf-8')
        contenido = archivo.read()
    except FileNotFoundError:
        return f"Error: archivo no encontrado: {ruta}"
    except PermissionError:
        return f"Error: sin permisos para leer: {ruta}"
    else:
        # Se ejecuta si NO hubo excepción
        return f"Leído correctamente ({len(contenido)} caracteres)"
    finally:
        # Se ejecuta SIEMPRE
        try:
            archivo.close()
            print("  (Archivo cerrado en finally)")
        except:
            pass

# Probar
print(leer_archivo(ruta_nueva))
print(leer_archivo("archivo_inexistente.txt"))


## 6. Lanzar excepciones (`raise`)


In [ ]:
# Lanzar una excepción cuando algo está mal
def dividir(a, b):
    if b == 0:
        raise ValueError("No se puede dividir por cero")
    return a / b

try:
    resultado = dividir(10, 0)
except ValueError as e:
    print(f"Error capturado: {e}")


In [ ]:
# Re-lanzar una excepción con contexto adicional
def procesar_pedido(pedido):
    if not pedido:
        raise ValueError("Pedido vacío")
    if "cliente" not in pedido:
        raise KeyError("Falta la clave 'cliente'")
    return f"Pedido procesado para {pedido['cliente']}"

try:
    procesar_pedido({"producto": "libro"})
except (ValueError, KeyError) as e:
    print(f"Error: {type(e).__name__}: {e}")


## 7. Excepciones personalizadas


In [ ]:
# Crear excepciones propias heredando de Exception
class SaldoInsuficienteError(Exception):
    """Excepción lanzada cuando no hay saldo suficiente."""
    def __init__(self, saldo, monto):
        self.saldo = saldo
        self.monto = monto
        super().__init__(f"Saldo insuficiente: ${saldo} < ${monto}")

class CuentaBancaria:
    def __init__(self, titular, saldo_inicial=0):
        self.titular = titular
        self.saldo = saldo_inicial
    
    def retirar(self, monto):
        if monto > self.saldo:
            raise SaldoInsuficienteError(self.saldo, monto)
        self.saldo -= monto
        return self.saldo

# Probar
cuenta = CuentaBancaria("Ana", 1000)
try:
    cuenta.retirar(1500)
except SaldoInsuficienteError as e:
    print(f"Error: {e}")
    print(f"Saldo actual: ${e.saldo}, monto solicitado: ${e.monto}")


## 8. Ejercicio Práctico: Procesador de Logs

**Contexto:** Vas a crear un script que lea un archivo de log, filtre las líneas que contienen errores, y guarde un reporte en un nuevo archivo.

**Instrucciones:**

Crea un archivo `procesador_logs.py` con:

1. `leer_logs(ruta)` — Lee un archivo de log línea por línea.
2. `filtrar_errores(lineas)` — Filtra líneas que contienen "ERROR" o "CRITICAL".
3. `generar_reporte(errores, ruta_salida)` — Guarda los errores en un archivo nuevo.
4. `procesar_log(ruta_entrada, ruta_salida)` — Función principal que combina todo.
5. Usa `with` para manejo seguro de archivos.
6. Usa try/except para manejar archivos inexistentes.

**Pistas:**
- Usa `with open(...) as f:` para abrir archivos.
- Para filtrar, usa `'ERROR' in linea or 'CRITICAL' in linea`.
- Maneja `FileNotFoundError` cuando el archivo no existe.

**Restricción del ejercicio:**
- No uses expresiones regulares (todavía).
- No crees clases (todavía).
- Solo usa funciones, `with`, `try/except`.


In [ ]:
"""
Solución al Ejercicio Práctico
Este código debe ir en un archivo procesador_logs.py
"""

def leer_logs(ruta):
    """Lee un archivo de log y retorna sus líneas."""
    try:
        with open(ruta, 'r', encoding='utf-8') as archivo:
            return archivo.readlines()
    except FileNotFoundError:
        print(f"Error: archivo no encontrado: {ruta}")
        return []
    except PermissionError:
        print(f"Error: sin permisos para leer: {ruta}")
        return []

def filtrar_errores(lineas):
    """Filtra líneas que contienen ERROR o CRITICAL."""
    errores = []
    for linea in lineas:
        if 'ERROR' in linea or 'CRITICAL' in linea:
            errores.append(linea)
    return errores

def generar_reporte(errores, ruta_salida):
    """Guarda los errores en un archivo nuevo."""
    with open(ruta_salida, 'w', encoding='utf-8') as archivo:
        archivo.write(f"=== REPORTE DE ERRORES ===\n")
        archivo.write(f"Total de errores: {len(errores)}\n")
        archivo.write("=" * 30 + "\n\n")
        for error in errores:
            archivo.write(error)

def procesar_log(ruta_entrada, ruta_salida):
    """Función principal: lee, filtra y guarda el reporte."""
    lineas = leer_logs(ruta_entrada)
    if not lineas:
        print("No se pudieron leer los logs.")
        return
    errores = filtrar_errores(lineas)
    if not errores:
        print("No se encontraron errores.")
        return
    generar_reporte(errores, ruta_salida)
    print(f"Reporte generado con {len(errores)} errores en: {ruta_salida}")


In [ ]:
# Probar el procesador de logs
import tempfile

# Crear un log de prueba
log_ejemplo = """2026-01-15 10:00:00 INFO Aplicación iniciada
2026-01-15 10:05:23 ERROR No se pudo conectar a la base de datos
2026-01-15 10:10:45 WARNING Memoria al 80%
2026-01-15 10:15:12 CRITICAL Sistema fuera de servicio
2026-01-15 10:20:33 INFO Reintentando conexión
2026-01-15 10:25:00 ERROR Timeout en la API externa
"""

ruta_log = os.path.join(tempfile.gettempdir(), "app.log")
with open(ruta_log, 'w', encoding='utf-8') as f:
    f.write(log_ejemplo)

# Procesar
ruta_reporte = os.path.join(tempfile.gettempdir(), "reporte_errores.txt")
procesar_log(ruta_log, ruta_reporte)

# Ver el reporte generado
print("\n--- Reporte generado ---")
with open(ruta_reporte, 'r', encoding='utf-8') as f:
    print(f.read())


## Resumen de la clase

En esta clase aprendimos:

1. **Apertura de archivos:** `open()` con modos `r`, `w`, `a`, `b`, `t`.
2. **Lectura:** `read()`, `readline()`, `readlines()`, iteración directa.
3. **Escritura:** `write()`, `writelines()`.
4. **`with`:** administrador de contexto para cierre automático.
5. **Excepciones:** `try/except/else/finally`.
6. **`raise`:** lanzar excepciones con mensajes claros.
7. **Excepciones personalizadas:** heredando de `Exception`.
8. **Filosofía "Fail Fast":** fallar pronto con mensajes descriptivos.

**En la siguiente clase** (Clase 13) veremos **POO Fundamental**: cómo crear clases, objetos, encapsulamiento y composición.
